# Memory with Function Calling

改用 function calling 讓 agent 自己決定要不要更新記憶，而不是每次對話結束才強制跑一個 memory update。

In [2]:
from pathlib import Path
from dotenv import load_dotenv
from litellm import completion
import json

load_dotenv()

MEMORY_FILE = Path("memory.md")

def load_memory() -> str:
    if not MEMORY_FILE.exists():
        return ""
    return MEMORY_FILE.read_text(encoding="utf-8").strip()

def save_memory(content: str):
    MEMORY_FILE.write_text(content, encoding="utf-8")

print("setup done")

setup done


## Tool 定義

給 agent 兩個工具：
- `update_memory`：用新內容完整覆蓋記憶（agent 自己負責整合舊記憶）
- `clear_memory`：清空所有記憶

In [3]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "update_memory",
            "description": (
                "當對話中出現值得長期記住的用戶資訊時呼叫此工具，"
                "例如：姓名、偏好、習慣、重要事項。"
                "傳入的 content 會完整覆蓋記憶檔案，所以要自己整合現有記憶與新資訊。"
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "content": {
                        "type": "string",
                        "description": "完整的記憶內容，用條列式 Markdown 格式"
                    }
                },
                "required": ["content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "clear_memory",
            "description": "清空所有記憶。只有在用戶明確要求時才呼叫。",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    }
]

# tool name -> 實際執行的函式
def handle_tool_call(name: str, arguments: dict) -> str:
    if name == "update_memory":
        save_memory(arguments["content"])
        print(f"[tool] update_memory called")
        return "記憶已更新。"
    elif name == "clear_memory":
        save_memory("")
        print(f"[tool] clear_memory called")
        return "記憶已清空。"
    return "unknown tool"

print("tools defined")

tools defined


## Chat loop with tool handling

當 model 決定呼叫 tool 時，執行 tool 並把結果回傳給 model，讓它繼續回覆用戶。

In [4]:
def new_conversation() -> list[dict]:
    memory = load_memory()
    system_content = (
        "你是一個有記憶的助理。"
        "當用戶透露值得記住的個人資訊時，主動呼叫 update_memory 工具保存。"
        "update_memory 的 content 需要整合現有記憶與新資訊，不要遺漏舊的內容。"
    )
    if memory:
        system_content += f"\n\n# 現有記憶\n{memory}"
    return [{"role": "system", "content": system_content}]


def chat(user_input: str, conversation: list[dict]) -> str:
    conversation.append({"role": "user", "content": user_input})

    while True:
        response = completion(
            model="gpt-4o-mini",
            messages=conversation,
            tools=TOOLS,
            tool_choice="auto",
        )

        message = response.choices[0].message

        # tool call 發生
        if message.tool_calls:
            # 把 assistant 的 tool_call message 加進 history
            conversation.append(message)

            # 執行每個 tool call 並回傳結果
            for tool_call in message.tool_calls:
                args = json.loads(tool_call.function.arguments)
                result = handle_tool_call(tool_call.function.name, args)

                conversation.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })
            # 繼續讓 model 根據 tool result 回覆
            continue

        # 沒有 tool call，正常回覆
        reply = message.content
        conversation.append({"role": "assistant", "content": reply})
        return reply


print("chat function loaded")

chat function loaded


## Demo：對話中 agent 自動決定更新記憶

In [5]:
# 清空舊記憶，重新開始
save_memory("")

convo = new_conversation()

print("User: 我叫小華，我是後端工程師，最近在學 LLM。")
reply = chat("我叫小華，我是後端工程師，最近在學 LLM。", convo)
print(f"Assistant: {reply}\n")

print("User: 我不喜歡用 JavaScript。")
reply = chat("我不喜歡用 JavaScript。", convo)
print(f"Assistant: {reply}\n")

print("User: 今天天氣真好。")
reply = chat("今天天氣真好。", convo)  # 這句不值得記憶，agent 應該不會呼叫 tool
print(f"Assistant: {reply}\n")

print("\n--- 目前 memory.md 內容 ---")
print(load_memory())

User: 我叫小華，我是後端工程師，最近在學 LLM。
[tool] update_memory called
Assistant: 已經記住你的資訊了！你叫小華，是後端工程師，最近在學習 LLM。如果還有其他想告訴我的事情，隨時可以說哦！

User: 我不喜歡用 JavaScript。
[tool] update_memory called
Assistant: 我已經記住了你不喜歡用 JavaScript。還有什麼想分享的嗎？

User: 今天天氣真好。
Assistant: 今天天氣好真讓人開心！希望你能享受這美好的一天。如果有什麼特別計劃，隨時可以和我分享！


--- 目前 memory.md 內容 ---
- 姓名: 小華
- 職業: 後端工程師
- 最近的學習主題: LLM
- 不喜歡的程式語言: JavaScript


## Demo：新的 session，驗證記憶

In [6]:
convo2 = new_conversation()

print("System prompt:")
print(convo2[0]["content"])
print()

print("User: 你記得我嗎？")
reply = chat("你記得我嗎？ 我不喜歡用什麼?", convo2)
print(f"Assistant: {reply}")

System prompt:
你是一個有記憶的助理。當用戶透露值得記住的個人資訊時，主動呼叫 update_memory 工具保存。update_memory 的 content 需要整合現有記憶與新資訊，不要遺漏舊的內容。

# 現有記憶
- 姓名: 小華
- 職業: 後端工程師
- 最近的學習主題: LLM
- 不喜歡的程式語言: JavaScript

User: 你記得我嗎？
Assistant: 當然！你不喜歡的程式語言是 JavaScript。還有其他你想更新或分享的資訊嗎？
